In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import os
import re
import time
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from DrissionPage import ChromiumPage, ChromiumOptions

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'MK NBMA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running MK NBMA Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

processdate = now.strftime('%Y-%m-%d')

# Lists from Jira DECD-6134 (ticket skips ListNr 2)
regdict = {
    1: {"ListName": "Banks",
        "URL": "https://www.nbrm.mk/banki-en.nspx",
        "Comments": "Extract all entities"},
    3: {"ListName": "Saving Houses",
        "URL": "https://www.nbrm.mk/stedilnici-en.nspx",
        "Comments": "Extract all entities"},
    4: {"ListName": "Other financial institutions",
        "URL": "https://www.nbrm.mk/ostanati_finansiski_institutcii-en.nspx",
        "Comments": "Extract all entities. Don't extract the entities under 'Government (non-financial) institutions'"},
    5: {"ListName": "National Bank of the Republic of North Macedonia participants",
        "URL": "https://www.nbrm.mk/registri-pups-en.nspx",
        "Comments": "Click the link 'Register of payment system operators' and extract the entities from the tab MIPS Participants"},
    6: {"ListName": "Clearing House - Clearing Interbank Systems AD Skopje participants",
        "URL": "https://www.nbrm.mk/registri-pups-en.nspx",
        "Comments": "Click the link 'Register of payment system operators' and extract the entities from the tab KIBS Participants"},
    7: {"ListName": "Register of payment institutions",
        "URL": "https://www.nbrm.mk/registri-pups-en.nspx",
        "Comments": "Click the link 'Register of payment institutions' and extract the entities"},
}

ListLabeldict = {1: 1, 3: 1, 4: 4, 5: 4, 6: 4, 7: 4}

BASE = 'https://www.nbrm.mk/'

In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# nbrm.mk is behind a Cloudflare JS challenge - plain requests gets 403.
# DrissionPage (real Chrome) passes the challenge automatically in a few seconds.
co = ChromiumOptions()
co.auto_port()
page = ChromiumPage(co)

def get_soup_dp(url, marker='inner-content-box', wait_cf=60):
    """Fetch url and wait until the Cloudflare challenge has cleared AND the
    expected content marker is present in the HTML (title alone is unreliable)."""
    page.get(url)
    for _ in range(wait_cf):
        html = page.html
        if 'just a moment' not in (page.title or '').lower() and marker in html:
            time.sleep(1)
            return BeautifulSoup(page.html, 'html.parser')
        time.sleep(1)
    raise RuntimeError(f'Cloudflare challenge not cleared or marker "{marker}" missing: {url} (title: {page.title})')

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def add_row(rowdata):
    for key in sqldict:
        sqldict[key].append(rowdata.get(key, ''))

def common_fields(listnr):
    return {'ListLabel': ListLabeldict[listnr],
            'RegCtry': 'MK',
            'RegCode': 'NBMA',
            'ListCode': str(listnr),
            'ListName': regdict[listnr]['ListName'],
            'ListLanguage': 'EN',
            'ListProcessDate': processdate,
            'RegulationType': 'Regulated',
            'Cntry': 'MK'}

def split_zip_city(address):
    """'Nikola Kljusev 1, 1000 Skopje' -> ('Nikola Kljusev 1', '1000', 'Skopje');
    addresses without the ', NNNN City' tail stay whole in Address_1."""
    m = re.search(r',\s*(\d{4})\s+(.+)$', address.strip())
    if m:
        return address[:m.start()].strip(), m.group(1), m.group(2).strip()
    return address.strip(), '', ''

In [6]:
#------------------------------------------------ Begin_Main : List 1 - Banks ----------------------------------------
listnr = 1
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

soup = get_soup_dp(regdict[listnr]['URL'])
box = soup.select_one('div.inner-content-box')
count = 0
for p in box.find_all('p', recursive=False):
    for a in p.find_all('a'):   # name from the anchor only - one <p> has trailing text outside the link
        row = common_fields(listnr)
        row.update({'Name': a.get_text(strip=True), 'Website': a.get('href', '')})
        add_row(row)
        count += 1

print(f"[INFO] : List 1 -> {count} entities")

[INFO] : Working _(Banks)_ 


[INFO] : List 1 -> 13 entities


In [7]:
#------------------------------------------------ Begin_Main : List 3 - Saving Houses ----------------------------------------
listnr = 3
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

soup = get_soup_dp(regdict[listnr]['URL'])
table = soup.select_one('div.inner-content-box table')
count = 0
for tr in table.find_all('tr')[1:]:   # row 0 = header; a trailing all-empty row is skipped below
    tds = tr.find_all('td')
    if len(tds) < 3 or not tds[1].get_text(strip=True):
        continue
    lines = [l.strip() for l in tds[1].get_text('\n').split('\n') if l.strip()]
    a = tds[1].find('a')
    phone = fax = email = ''
    for l in lines[2:]:
        ll = l.lower()
        if ll.startswith('phone'):
            phone = l.split(':', 1)[1].strip()
        elif ll.startswith(('fax', 'faks')):   # site uses Macedonian spelling 'Faks' on one row
            fax = l.split(':', 1)[1].strip()
        elif ll.startswith('e-mail'):
            email = l.split(':', 1)[1].strip()
    addr1, zipc, city = split_zip_city(lines[1] if len(lines) > 1 else '')
    row = common_fields(listnr)
    row.update({'Name': lines[0], 'Address_1': addr1, 'Zip': zipc, 'City': city,
                'Phone': phone, 'Fax': fax, 'Email': email,
                'Website': a.get('href', '') if a else ''})
    add_row(row)
    count += 1

print(f"[INFO] : List 3 -> {count} entities")

[INFO] : Working _(Saving Houses)_ 


[INFO] : List 3 -> 2 entities


In [8]:
#------------------------------------------------ Begin_Main : List 4 - Other financial institutions ----------------------------------------
listnr = 4
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

soup = get_soup_dp(regdict[listnr]['URL'])
p = soup.select_one('p#ArticleTitle').find_next_sibling('p')
count = 0
# entities are <a> tags; the excluded 'Government (non-financial) institutions' section
# starts at a <strong> marker and its entries are plain text (no <a>) - stop at the marker
for child in p.children:
    if child.name == 'strong':
        break
    if child.name == 'a':
        row = common_fields(listnr)
        row.update({'Name': child.get_text(strip=True), 'Website': child.get('href', '')})
        add_row(row)
        count += 1

print(f"[INFO] : List 4 -> {count} entities")

[INFO] : Working _(Other financial institutions)_ 


[INFO] : List 4 -> 3 entities


In [9]:
#------------------------------------------------ Begin_Main : registers page - download the two xlsx ----------------------------------------
soup = get_soup_dp('https://www.nbrm.mk/registri-pups-en.nspx')

xlsx_paths = {}
for label in ['Register of payment system operators', 'Register of payment institutions']:
    link = None
    for a in soup.select('div.inner-content-box a'):
        if a.get_text(' ', strip=True) == label:   # anchors contain an <img>, so string= match fails
            link = a
            break
    if link is None:
        raise RuntimeError(f'link not found on registers page: {label}')
    url = urljoin(BASE, link['href'])   # operators filename is version-stamped (_12026) - never hardcode
    # reuse the Cloudflare clearance: browser cookies + the browser's own UA (cf_clearance is UA-bound)
    headers = {'User-Agent': page.run_js('return navigator.userAgent'),
               'Referer': 'https://www.nbrm.mk/registri-pups-en.nspx'}
    r = requests.get(url, headers=headers, cookies=page.cookies().as_dict(), timeout=60)
    r.raise_for_status()
    local = os.path.join(tempfolder, label.replace(' ', '_') + '.xlsx')
    with open(local, 'wb') as f:
        f.write(r.content)
    xlsx_paths[label] = local
    print(f"[INFO] : downloaded {label} -> {local} ({len(r.content)} bytes)")

[INFO] : downloaded Register of payment system operators -> /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/MK NBMA/tempfolder/Register_of_payment_system_operators.xlsx (112568 bytes)
[INFO] : downloaded Register of payment institutions -> /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/MK NBMA/tempfolder/Register_of_payment_institutions.xlsx (11074 bytes)


In [10]:
#------------------------------------------------ Begin_Main : Lists 5 & 6 - MIPS / KIBS participants ----------------------------------------
operators_xlsx = xlsx_paths['Register of payment system operators']

for listnr, sheet in [(5, 'MIPS Participants'), (6, 'KIBS Participants')]:
    print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")
    df = pd.read_excel(operators_xlsx, sheet_name=sheet, header=2, dtype=str, keep_default_na=False)
    # layout: col 0 spacer, col 1 Ref. no., col 2 Name, col 3 UCIN, col 4 Address
    df = df[df.iloc[:, 2].str.strip() != '']
    for _, r in df.iterrows():
        name = r.iloc[2].strip()
        ucin = r.iloc[3].strip()
        if ucin.upper() == 'X':   # placeholder for foreign participants without a UCIN (e.g. MasterCard)
            ucin = ''
        addr1, zipc, city = split_zip_city(r.iloc[4])
        row = common_fields(listnr)
        row.update({'Name': name, 'Address_1': addr1, 'Zip': zipc, 'City': city,
                    'InternalID_1': ucin, 'InternalID_1_type': 'UCIN' if ucin else ''})
        add_row(row)
    print(f"[INFO] : List {listnr} -> {len(df)} entities")

[INFO] : Working _(National Bank of the Republic of North Macedonia participants)_ 


[INFO] : List 5 -> 19 entities
[INFO] : Working _(Clearing House - Clearing Interbank Systems AD Skopje participants)_ 
[INFO] : List 6 -> 14 entities


In [11]:
#------------------------------------------------ Begin_Main : List 7 - Register of payment institutions ----------------------------------------
listnr = 7
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

df = pd.read_excel(xlsx_paths['Register of payment institutions'], header=2, dtype=str, keep_default_na=False)
df = df[df.iloc[:, 1].str.strip() != '']   # col 1 = Title
for _, r in df.iterrows():
    # cols: 0 No. | 1 Title | 2 Single tax number | 3 UCIN | 4 Main offfice (sic) | 5 Phone | 6 E-mail
    #       7 Website | 8 licensing decision 'DD.MM.YYYY / D No. ...' | 9 services | ... | 13 status
    status = r.iloc[13].strip().lower()
    if status and status != 'active':
        print(f"[WARN] : '{r.iloc[1].strip()}' status is '{status}' (not active) - review RegulationType")
    m = re.search(r'(\d{2})\.(\d{2})\.(\d{4})', r.iloc[8])
    regdate = f"{m.group(3)}-{m.group(2)}-{m.group(1)}" if m else ''
    row = common_fields(listnr)
    row.update({'Name': r.iloc[1].strip(),
                'InternalID_1': r.iloc[2].strip(), 'InternalID_1_type': 'Tax number' if r.iloc[2].strip() else '',
                'InternalID_2': r.iloc[3].strip(), 'InternalID_2_type': 'UCIN' if r.iloc[3].strip() else '',
                'Address_1': r.iloc[4].strip(), 'Phone': r.iloc[5].strip(),
                'Email': r.iloc[6].strip(), 'Website': r.iloc[7].strip(),
                'RegulationDate': regdate, 'License_Type': r.iloc[9].strip()})
    add_row(row)

print(f"[INFO] : List 7 -> {len(df)} entities")

[INFO] : Working _(Register of payment institutions)_ 
[INFO] : List 7 -> 5 entities


In [12]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
page.quit()

os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))

Saved 56 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/MK NBMA/MK NBMA SQL Ready 2026-07-13 10.17.56.xlsx
